## Veri Setini Yükleme

In [ ]:
TRAIN_DIR ="Training"
TEST_DIR  = "Testing"

labels = ["glioma", "meningioma", "notumor", "pituitary"]
IMAGE_SIZE = 64
SEED = 42

In [ ]:
def load_images(directory, image_size):
    X, y = [], []

    for label in labels:
        folder = os.path.join(directory, label)
        if not os.path.exists(folder):
            print(f"{folder} bulunamadı!")
            continue

        for img_name in tqdm(os.listdir(folder), desc=f"Yükleniyor: {label}"):
            img_path = os.path.join(folder, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (image_size, image_size))
            img = img / 255.0  # normalize

            X.append(img)
            y.append(labels.index(label))

    X = np.array(X, dtype=np.float32)
    y = np.array(y)

    return shuffle(X, y, random_state=SEED)

In [ ]:
print("Training verisi yükleniyor...")
X_train, y_train = load_images(TRAIN_DIR, IMAGE_SIZE)

print("\nTesting verisi yükleniyor...")
X_test, y_test = load_images(TEST_DIR, IMAGE_SIZE)

print("\nYükleme tamamlandı!")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## Flatten İşlemi

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
X_train= X_train.reshape(X_train.shape[0], -1)
X_test= X_test.reshape(X_test.shape[0], -1)

In [ ]:
print(X_train.shape)
print(y_train.shape)

## Logistic Regression Hold Out

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=5000,
    solver='lbfgs',
    n_jobs=-1
)

In [ ]:
lr.fit(X_train, y_train)

In [ ]:
y_pred_lr=lr.predict(X_test)
y_prob_lr=lr.predict_proba(X_test)

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", round(accuracy_lr, 4))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

In [ ]:
report = classification_report(
    y_test,
    y_pred_lr,
    target_names=labels,
    output_dict=True
)

df = pd.DataFrame(report).T
df = df.iloc[:-3][['precision', 'recall', 'f1-score']]

specificity = []
cm = cm_lr

for i in range(len(labels)):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    spec = TN / (TN + FP)
    specificity.append(spec)

df['specificity'] = specificity
df = df.round(4)

print("Logistic Regression - Performans Tablosu")
print("\nAccuracy:", round(accuracy_lr, 4))
print("-"*45)
print(df)

In [ ]:
plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_lr[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{label} (AUC = {auc_score:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression - ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

## Logistic Regression K-Fold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
all_true = []
all_pred = []

print("Logistic Regression - k-Fold Sonuçları")
print("-" * 45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k, X_val_k = X_train[train_idx], X_train[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    lr = LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        n_jobs=-1
    )

    lr.fit(X_train_k, y_train_k)
    y_pred_k = lr.predict(X_val_k)

    # Accuracy
    acc = accuracy_score(y_val_k, y_pred_k)
    fold_accuracies.append(acc)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

print("-" * 45)
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std Sapma        : {np.std(fold_accuracies):.4f}")

In [ ]:
metrics = {
    "precision": [],
    "recall": [],
    "f1-score": [],
    "specificity": []
}

for i, label in enumerate(labels):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    f1        = 2 * precision * recall / (precision + recall)
    specificity = TN / (TN + FP)

    metrics["precision"].append(precision)
    metrics["recall"].append(recall)
    metrics["f1-score"].append(f1)
    metrics["specificity"].append(specificity)

df_kfold_metrics = pd.DataFrame(metrics, index=labels).round(4)

print("Logistic Regression - k-Fold Performans Tablosu")
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print("-" * 45)
print(df_kfold_metrics)

In [ ]:
cm = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Logistic Regression - k-Fold Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:

class_names = ["glioma", "meningioma", "notumor", "pituitary"]
n_classes = len(class_names)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_true_all = []
y_prob_all = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k, X_val_k = X_train[train_idx], X_train[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    lr = LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        n_jobs=-1
    )

    lr.fit(X_train_k, y_train_k)

    y_prob = lr.predict_proba(X_val_k)

    y_true_all.append(y_val_k)
    y_prob_all.append(y_prob)

y_true_all = np.concatenate(y_true_all)
y_prob_all = np.concatenate(y_prob_all)


y_true_bin = label_binarize(y_true_all, classes=range(n_classes))

plt.figure(figsize=(8, 6))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob_all[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Logistic Regression - k-Fold ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

## KNN HOLD OUT

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_knn = scaler.fit_transform(X_train)
X_test_knn  = scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="euclidean"
)

knn.fit(X_train_knn, y_train)

In [ ]:
y_pred_knn=knn.predict(X_test_knn)
y_prob_knn=knn.predict_proba(X_test_knn)

In [ ]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)

cm_knn= confusion_matrix(y_test, y_pred_knn)

metrics = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    specificity = TN / (TN + FP)
    f1 = 2 * (precision * recall) / (precision + recall)

    metrics.append([
        precision,
        recall,
        f1,
        specificity
    ])

metrics_df = pd.DataFrame(
    metrics,
    index=labels,
    columns=["precision", "recall", "f1-score", "specificity"]
)

metrics_df = metrics_df.round(4)

print("KNN - Hold-Out Performans Tablosu")
print("-" * 45)
print(metrics_df)

print(f"\nGenel Doğruluk (Accuracy): {accuracy_knn:.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("KNN - Hold-Out Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_knn[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN - Hold-Out ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

## KNN K-FOLD

In [ ]:
scaler = StandardScaler()

X_train_knn = scaler.fit_transform(X_train) 
X_test_knn  = scaler.transform(X_test)


joblib.dump(scaler, "scaler2.pkl")


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies_knn = []
best_acc = 0.0
best_knn_model = None


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_knn, y_train), 1):

    X_train_k, X_val_k = X_train_knn[train_idx], X_train_knn[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_k, y_train_k)
    
    y_pred_fold = knn.predict(X_val_k)
    acc = accuracy_score(y_val_k, y_pred_fold)
    
    fold_accuracies_knn.append(acc)

    if acc > best_acc:
        best_acc = acc
        best_knn_model = knn

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

joblib.dump(best_knn_model, "knn_model2.pkl") 

print(f"\nOrtalama Accuracy: {np.mean(fold_accuracies_knn):.4f}")
print("En iyi model 'knn_model2.pkl' olarak kaydedildi.")

In [ ]:
cm = confusion_matrix(y_test,y_pred_knn)

rows = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    f1        = 2 * precision * recall / (precision + recall)
    specificity = TN / (TN + FP)

    rows.append([
        precision,
        recall,
        f1,
        specificity
    ])

metrics_table_knn = pd.DataFrame(
    rows,
    index=labels,
    columns=["precision", "recall", "f1-score", "specificity"]
)

accuracy_knn = accuracy_score(y_test, y_pred_knn)

print("KNN - K-Fold Performans Tablosu")
print("-" * 45)
print(metrics_table_knn.round(4))
print("\nGenel Doğruluk (Accuracy): {:.4f}".format(accuracy_knn))

In [ ]:
all_true = []
all_pred = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_k, y_train_k)

    y_pred_k = knn.predict(X_val_k)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)

cm_knn_kfold = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_knn_kfold,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("KNN - k-Fold Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
all_true = np.array(all_true)
all_prob = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_k, y_train_k)

    probs = knn.predict_proba(X_val_k)
    all_prob.append(probs)

all_prob = np.vstack(all_prob)

plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true == i, all_prob[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN - k-Fold ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

## SVM HOLD OUT

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

svm.fit(X_train, y_train)

In [ ]:
y_pred_svm=svm.predict(X_test)
y_prob_svm=svm.predict_proba(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred_svm)

rows = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    f1        = 2 * precision * recall / (precision + recall)
    specificity = TN / (TN + FP)

    rows.append([precision, recall, f1, specificity])

metrics_table_svm = pd.DataFrame(
    rows,
    index=labels,
    columns=["precision", "recall", "f1-score", "specificity"]
)

accuracy_svm = accuracy_score(y_test, y_pred_svm)

print("SVM - Hold-Out Performans Tablosu")
print("-" * 45)
print(metrics_table_svm.round(4))
print("\nGenel Doğruluk (Accuracy): {:.4f}".format(accuracy_svm))

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("SVM - Hold-Out Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_svm[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM - Hold-Out ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

## SVM K-FOLD

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []

all_true = []
all_pred = []
all_prob = []


for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    svm = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=42
    )

    svm.fit(X_train_k, y_train_k)

    y_pred_k = svm.predict(X_val_k)
    y_prob_k = svm.predict_proba(X_val_k)

    acc = accuracy_score(y_val_k, y_pred_k)
    fold_accuracies.append(acc)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)
    all_prob.extend(y_prob_k)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

In [ ]:
rows = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)

    precision = TP / (TP + FP)
    recall    = TP / (TP + FN)
    f1        = 2 * precision * recall / (precision + recall)
    specificity = TN / (TN + FP)

    rows.append([precision, recall, f1, specificity])

metrics_table_svm_kfold = pd.DataFrame(
    rows,
    index=labels,
    columns=["precision", "recall", "f1-score", "specificity"]
)

print("SVM - k-Fold Performans Tablosu")
print("-" * 45)
print(metrics_table_svm_kfold.round(4))

In [ ]:
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt

cm = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("SVM - k-Fold Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
all_prob = np.array(all_prob)
all_true = np.array(all_true)

plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true == i, all_prob[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM - k-Fold ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


## Naive Bayes Hold Out

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)

In [ ]:
y_pred_nb=nb.predict(X_test)
y_prob_nb=nb.predict_proba(X_test)

In [ ]:
acc_nb = accuracy_score(y_test, y_pred_nb)

report = classification_report(
    y_test,
    y_pred_nb,
    target_names=labels,
    output_dict=True
)

cm = cm_nb
specificity = {}

for i, label in enumerate(labels):
    TN = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    FP = cm[:, i].sum() - cm[i, i]
    specificity[label] = TN / (TN + FP)

table = pd.DataFrame({
    "precision": [report[l]["precision"] for l in labels],
    "recall":    [report[l]["recall"] for l in labels],
    "f1-score":  [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("Naive Bayes - Hold-Out Performans Tablosu")
print("-"*45)
print(table.round(4))

print(f"\nGenel Doğruluk (Accuracy): {acc_nb:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix

cm_nb=confusion_matrix(y_test,y_pred_nb)


labels = ["glioma", "meningioma", "notumor", "pituitary"]

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_nb,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Naive Bayes - Hold-Out Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_nb[:, i])
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Naive Bayes - Hold-Out ROC Eğrileri")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

## Naive Bayes K-Fold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []

print("Naive Bayes - k-Fold Sonuçları")
print("-" * 45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    nb = GaussianNB()
    nb.fit(X_train_k, y_train_k)

    y_pred_k = nb.predict(X_val_k)
    acc = accuracy_score(y_val_k, y_pred_k)
    fold_accuracies.append(acc)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

print("-" * 45)
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std Sapma: {np.std(fold_accuracies):.4f}")

In [ ]:
all_true = []
all_pred = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    nb = GaussianNB()
    nb.fit(X_train_k, y_train_k)

    y_pred_k = nb.predict(X_val_k)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)

cm_nb_kfold = confusion_matrix(all_true, all_pred)

In [ ]:
report = classification_report(
    all_true,
    all_pred,
    target_names=labels,
    output_dict=True
)

specificity = {}
cm = cm_nb_kfold

for i, label in enumerate(labels):
    TN = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    FP = cm[:, i].sum() - cm[i, i]
    specificity[label] = TN / (TN + FP)

table_nb_kfold = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("Naive Bayes - k-Fold Performans Tablosu")
print("-"*45)
print(table_nb_kfold.round(4))

print(f"\nGenel Doğruluk (Accuracy): {np.mean(fold_accuracies):.4f}")

In [ ]:
labels = ["glioma", "meningioma", "notumor", "pituitary"]

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_nb_kfold,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Naive Bayes - k-Fold Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true == i, all_probs[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Naive Bayes - ROC Curve (k-Fold)")
plt.legend()
plt.show()

## Decision Tree Hold Out

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42,
    max_depth=None
)

In [ ]:
dt.fit(X_train,y_train)

In [ ]:
y_pred_dt=dt.predict(X_test)
y_prob_dt=dt.predict_proba(X_test)

In [ ]:
acc = accuracy_score(y_test,y_pred_dt)


labels = ["glioma", "meningioma", "notumor", "pituitary"]

cm_dt = confusion_matrix(y_test, y_pred_dt)

report = classification_report(
    y_test,
    y_pred_dt,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_dt.sum() - (cm_dt[i, :].sum() + cm_dt[:, i].sum() - cm_dt[i, i])
    FP = cm_dt[:, i].sum() - cm_dt[i, i]
    specificity[label] = TN / (TN + FP)

table_dt = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nDecision Tree - Hold-Out Performans Tablosu")
print("-"*45)
print(table_dt.round(4))

In [ ]:
plt.figure(figsize=(7,6))
sns.heatmap(
    cm_dt,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Decision Tree - Hold-Out Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_dt[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Decision Tree - ROC Curve (Hold-Out)")
plt.legend()
plt.show()

## Decision Tree K-Fold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies_dt = []

print("Decision Tree - k-Fold Sonuçları")
print("-"*45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    dt = DecisionTreeClassifier(
        random_state=42
    )

    dt.fit(X_train_k, y_train_k)
    y_pred_k = dt.predict(X_val_k)

    acc = accuracy_score(y_val_k, y_pred_k)
    fold_accuracies_dt.append(acc)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

print("-"*45)
print(f"Ortalama Accuracy: {np.mean(fold_accuracies_dt):.4f}")
print(f"Std Sapma: {np.std(fold_accuracies_dt):.4f}")

In [ ]:
report = classification_report(
    all_true_dt,
    all_pred_dt,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_dt_kfold.sum() - (cm_dt_kfold[i, :].sum() + cm_dt_kfold[:, i].sum() - cm_dt_kfold[i, i])
    FP = cm_dt_kfold[:, i].sum() - cm_dt_kfold[i, i]
    specificity[label] = TN / (TN + FP)

table_dt_kfold = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nDecision Tree - k-Fold Performans Tablosu")
print("Accuarcy: ",acc)
print("-"*50)
print(table_dt_kfold.round(4))

In [ ]:
all_true_dt = []
all_pred_dt = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train_k, y_train_k)

    y_pred_k = dt.predict(X_val_k)

    all_true_dt.extend(y_val_k)
    all_pred_dt.extend(y_pred_k)

cm_dt_kfold = confusion_matrix(all_true_dt, all_pred_dt)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_dt_kfold,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Decision Tree - k-Fold Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

all_true_dt = np.array(all_true_dt)
all_pred_prob_dt = []

for train_idx, val_idx in skf.split(X_train, y_train):

    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train[train_idx], y_train[train_idx])

    probs = dt.predict_proba(X_train[val_idx])
    all_pred_prob_dt.append(probs)

all_pred_prob_dt = np.vstack(all_pred_prob_dt)

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true_dt == i, all_pred_prob_dt[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Decision Tree - k-Fold ROC Curve")
plt.legend()
plt.show()

## Ensemble Learning

## Random Forest Hold Out

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf=RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
y_pred_rf=rf.predict(X_test)
y_prob_rf=rf.predict_proba(X_test)

In [ ]:
accuarcy_rf=accuracy_score(y_test,y_pred_rf)

report = classification_report(
    y_test,
    y_pred_rf,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_rf.sum() - (cm_rf[i, :].sum() + cm_rf[:, i].sum() - cm_rf[i, i])
    FP = cm_rf[:, i].sum() - cm_rf[i, i]
    specificity[label] = TN / (TN + FP)

rf_table = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nRandom Forest - Hold-Out Performans Tablosu")
print("Accuarcy: ",accuarcy_rf)
print("-"*50)
print(rf_table.round(4))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen Sınıf")
plt.ylabel("Gerçek Sınıf")
plt.title("Random Forest - Hold-Out Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_rf[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest - Hold-Out ROC Curve")
plt.legend()
plt.show()    

## Random Forest K-Fold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
best_acc = 0.0
best_rf_model = None

print("Random Forest - 5-Fold Eğitim Başlatılıyor...")
print("-" * 45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_train_k, X_val_k = X_train[train_idx], X_train[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,   
        random_state=42,      
        n_jobs=-1,        
        class_weight='balanced'
    )


    rf.fit(X_train_k, y_train_k)
    

    y_pred = rf.predict(X_val_k)
    acc = accuracy_score(y_val_k, y_pred)
    
    fold_accuracies.append(acc)


    if acc > best_acc:
        best_acc = acc
        best_rf_model = rf

    print(f"Fold {fold}: Başarı Oranı (Accuracy) = {acc:.4f}")

print("-" * 45)
print(f"Ortalama Başarı: {np.mean(fold_accuracies):.4f}")
print(f"En İyi Fold Başarısı: {best_acc:.4f}")


joblib.dump(best_rf_model, "rf_model.pkl")

print("\nİşlem Tamamlandı: 'rf_model.pkl' başarıyla kaydedildi.")

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []
all_y_true_kfold = [] 
all_y_pred_rf_kfold = []

print("Random Forest - k-Fold Sonuçları")
print("-"*45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]
    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_k, y_train_k)
    y_pred_rf_knn = rf.predict(X_val_k)

    acc = accuracy_score(y_val_k, y_pred_rf_knn)
    fold_accuracies.append(acc)

    all_y_true_kfold.extend(y_val_k)
    all_y_pred_rf_kfold.extend(y_pred_rf_knn)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

print("-"*45)
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std Sapma: {np.std(fold_accuracies):.4f}")

np.save('kfold_y_true.npy', np.array(all_y_true_kfold))
np.save('kfold_y_pred_rf.npy', np.array(all_y_pred_rf_kfold))

In [ ]:
report = classification_report(
    all_true,
    all_pred,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_rf_kfold.sum() - (cm_rf_kfold[i, :].sum() + cm_rf_kfold[:, i].sum() - cm_rf_kfold[i, i])
    FP = cm_rf_kfold[:, i].sum() - cm_rf_kfold[i, i]
    specificity[label] = TN / (TN + FP)

rf_kfold_table = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nRandom Forest - k-Fold Performans Tablosu")
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print("-"*55)
print(rf_kfold_table.round(4))

In [ ]:
all_true = []
all_pred = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_k, y_train_k)
    y_pred_k = rf.predict(X_val_k)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)

cm_rf_kfold = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_rf_kfold,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek")
plt.title("Random Forest - k-Fold Confusion Matrix")
plt.show()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_true = []
all_prob = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k, X_val_k = X_train[train_idx], X_train[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_k, y_train_k)
    y_prob_k = rf.predict_proba(X_val_k)

    all_true.extend(y_val_k)
    all_prob.extend(y_prob_k)

all_true = np.array(all_true)
all_prob = np.array(all_prob)

# ROC ÇİZİMİ
plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true == i, all_prob[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Random Forest - k-Fold ROC Curve")
plt.legend()
plt.grid()
plt.show()

## AdaBoost Hold Out

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

In [ ]:
ada.fit(X_train,y_train)

In [ ]:
y_pred_ada=ada.predict(X_test)
y_prob_ada=ada.predict_proba(X_test)

In [ ]:
accuarcy_ada = accuracy_score(y_test,y_pred_ada)
print(accuarcy_ada)

In [ ]:
cm_ada=confusion_matrix(y_test,y_pred_ada)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_ada,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek")
plt.title("AdaBoost - Hold-Out Confusion Matrix")
plt.show()

In [ ]:
report = classification_report(
    y_test,
    y_pred_ada,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_ada.sum() - (cm_ada[i, :].sum() + cm_ada[:, i].sum() - cm_ada[i, i])
    FP = cm_ada[:, i].sum() - cm_ada[i, i]
    specificity[label] = TN / (TN + FP)

ada_table = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nAdaBoost - Hold-Out Performans Tablosu")
print("-"*55)
print(ada_table.round(4))
print("\nGenel Doğruluk (Accuracy):", round(accuarcy_ada, 4))

In [ ]:
plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_test == i, y_prob_ada[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("AdaBoost - Hold-Out ROC Curve")
plt.legend()
plt.show()

## AdaBoost K-Fold

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import AdaBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_accuracies = []

print("AdaBoost - k-Fold Sonuçları")
print("-"*45)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    ada = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    )

    ada.fit(X_train_k, y_train_k)
    y_pred_k = ada.predict(X_val_k)

    acc = accuracy_score(y_val_k, y_pred_k)
    fold_accuracies.append(acc)

    print(f"Fold {fold}: Accuracy = {acc:.4f}")

print("-"*45)
print(f"Ortalama Accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std Sapma: {np.std(fold_accuracies):.4f}")

In [ ]:
all_true = []
all_pred = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k = X_train[train_idx]
    X_val_k   = X_train[val_idx]

    y_train_k = y_train[train_idx]
    y_val_k   = y_train[val_idx]

    ada = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    )

    ada.fit(X_train_k, y_train_k)
    y_pred_k = ada.predict(X_val_k)

    all_true.extend(y_val_k)
    all_pred.extend(y_pred_k)

cm_ada_kfold = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(7,6))
sns.heatmap(
    cm_ada_kfold,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gerçek")
plt.title("AdaBoost - k-Fold Confusion Matrix")
plt.show()

In [ ]:
report = classification_report(
    all_true,
    all_pred,
    target_names=labels,
    output_dict=True
)

specificity = {}

for i, label in enumerate(labels):
    TN = cm_ada_kfold.sum() - (cm_ada_kfold[i, :].sum() + cm_ada_kfold[:, i].sum() - cm_ada_kfold[i, i])
    FP = cm_ada_kfold[:, i].sum() - cm_ada_kfold[i, i]
    specificity[label] = TN / (TN + FP)

ada_kfold_table = pd.DataFrame({
    "precision":   [report[l]["precision"] for l in labels],
    "recall":      [report[l]["recall"] for l in labels],
    "f1-score":    [report[l]["f1-score"] for l in labels],
    "specificity": [specificity[l] for l in labels]
}, index=labels)

print("\nAdaBoost - k-Fold Performans Tablosu")
print(f"Accuracy: {np.mean(fold_accuracies):.4f}")
print("-"*55)
print(ada_kfold_table.round(4))

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import AdaBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_true = []
all_prob = []

for train_idx, val_idx in skf.split(X_train, y_train):

    X_train_k, X_val_k = X_train[train_idx], X_train[val_idx]
    y_train_k, y_val_k = y_train[train_idx], y_train[val_idx]

    ada = AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    )

    ada.fit(X_train_k, y_train_k)
    y_prob_k = ada.predict_proba(X_val_k)

    all_true.extend(y_val_k)
    all_prob.extend(y_prob_k)

all_true = np.array(all_true)
all_prob = np.array(all_prob)

plt.figure(figsize=(7,6))

for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(all_true == i, all_prob[:, i])
    auc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc_score:.3f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("AdaBoost - k-Fold ROC Curve")
plt.legend()
plt.grid()
plt.show()

## McNemar Test

In [ ]:
np.save("rf_y_true.npy", y_test)
np.save("rf_y_pred.npy", y_pred_rf)

In [ ]:
y_true_cnn = np.load("cnn_y_true.npy")
y_pred_cnn = np.load("cnn_y_pred.npy")
y_true_rf = np.load("rf_y_true.npy")

In [ ]:

y_true_actual = np.load('cnn_y_true.npy')
y_pred_cnn = np.load('cnn_y_pred.npy')
print("CNN dosyaları başarıyla yüklendi.")


table = mcnemar_table(y_target=y_true_actual, 
                      y_model1=y_pred_cnn, 
                      y_model2=y_pred_rf)

print("\n--- McNemar Kontenjans Tablosu (CNN vs RF) ---")
print(table)


chi2, p = mcnemar(ary=table, corrected=True)
print(f"\np-değeri: {p:.4f}")

df_table = pd.DataFrame(table, 
                        index=['CNN Doğru', 'CNN Yanlış'], 
                        columns=['RF Doğru', 'RF Yanlış'])

print("--- Akademik McNemar Tablosu ---")
print(df_table)
print(f"\np-değeri: {p:.4f}")


In [ ]:
table = np.array([[95, 205], 
                  [244, 767]])

chi2, p = mcnemar(ary=table, corrected=True)


df_table = pd.DataFrame(table, 
                        index=['CNN Doğru', 'CNN Yanlış'], 
                        columns=['RF Doğru', 'RF Yanlış'])

print("---McNemar Tablosu ---")
print(df_table)
print(f"\np-değeri: {p:.4f}")

plt.figure(figsize=(8, 6))
sns.heatmap(df_table, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 16})
plt.title(f'McNemar Testi: CNN vs Random Forest\n(p-value: {p:.4f})', fontsize=15, pad=20)
plt.xlabel('Random Forest Tahminleri', fontsize=12)
plt.ylabel('EfficientNetB0 (CNN) Tahminleri', fontsize=12)

## McNemar K-Fold

In [ ]:
from mlxtend.evaluate import mcnemar_table, mcnemar


y_true_k = np.load('kfold_y_true.npy')
y_rf_k = np.load('kfold_y_pred_rf.npy')
y_knn_k = np.load('kfold_y_pred_knn.npy')


table_kfold = mcnemar_table(y_target=y_true_k, 
                            y_model1=y_rf_k, 
                            y_model2=y_knn_k)


chi2, p = mcnemar(ary=table_kfold, corrected=True)


df_kfold = pd.DataFrame(table_kfold, 
                        index=['RF Doğru', 'RF Yanlış'], 
                        columns=['KNN Doğru', 'KNN Yanlış'])

plt.figure(figsize=(8, 6))
sns.heatmap(df_kfold, annot=True, fmt='d', cmap='YlGnBu', cbar=False, annot_kws={"size": 16})
plt.title(f'k-Fold McNemar Analizi: Random Forest vs KNN\n(p-value: {p:.4f})', fontsize=14)
plt.xlabel('KNN Tahminleri')
plt.ylabel('Random Forest Tahminleri')
plt.show()

print(f"p-değeri: {p:.4f}")